# ACT Creative 网站风险、SEO 与 AI 搜索审计

数据复核日期：2026-08-12（Asia/Singapore）。Search Console 最近 28 天窗口为 2026-07-13 至 2026-08-09，前期为 2026-06-15 至 2026-07-12。

## tl;dr

- 生产部署来自 `codex/Christmas-NewYear-CNY@151eb3c`，本地 `master@f5a69cd` 落后；在任何修改前应先安全对齐。
- `npm audit --omit=dev` 发现 `undici 7.28.0` 的 1 个高危漏洞，修复版本为 `>=7.29.0`。
- GSC 点击和曝光均增长约 71%，CTR 仍约 0.9%；AI impressions 从 218 增至 570。

## Context & Methods

输入包括 GSC Search results 与 AI report、Vercel 部署元数据、生产 HTTP 响应头、本地仓库、`npm audit`、`npm run seo:check` 和 `npm run build`。标准搜索 impressions 与 AI impressions 定义不同，不做加总。

In [1]:
gsc = {
    'clicks_current': 96, 'clicks_prior': 56,
    'impressions_current': 10751, 'impressions_prior': 6269,
    'position_current': 28.5, 'position_prior': 32.2,
    'ai_current': 570, 'ai_prior': 218,
}
pages = [
    {'page': 'Event fabrication SG', 'clicks': 6, 'impressions': 1108, 'ctr': 0.005, 'position': 10.7},
    {'page': 'National Museum', 'clicks': 0, 'impressions': 69, 'ctr': 0.0, 'position': 7.0},
    {'page': 'Venue finder', 'clicks': 4, 'impressions': 290, 'ctr': 0.014, 'position': 7.6},
    {'page': 'Marina Bay Sands', 'clicks': 4, 'impressions': 485, 'ctr': 0.008, 'position': 11.0},
    {'page': 'Homepage / brand', 'clicks': 9, 'impressions': 84, 'ctr': 0.107, 'position': 5.9},
    {'page': 'Booth design/build', 'clicks': 9, 'impressions': 4620, 'ctr': 0.002, 'position': 38.1},
]
ai_pages = {'Booth design/build': 140, 'Event fabrication SG': 139, 'Homepage': 44, 'Custom props': 41, 'Food truck': 35, 'Venue finder': 29}

## Results

In [2]:
click_growth = gsc['clicks_current'] / gsc['clicks_prior'] - 1
impression_growth = gsc['impressions_current'] / gsc['impressions_prior'] - 1
ctr_current = gsc['clicks_current'] / gsc['impressions_current']
ctr_prior = gsc['clicks_prior'] / gsc['impressions_prior']
ai_growth = gsc['ai_current'] / gsc['ai_prior'] - 1
print(f"Clicks growth: {click_growth:.2%}")
print(f"Impressions growth: {impression_growth:.2%}")
print(f"CTR current/prior: {ctr_current:.4%} / {ctr_prior:.4%}")
print(f"Position improvement: {gsc['position_prior'] - gsc['position_current']:.1f}")
print(f"AI impressions growth: {ai_growth:.2%}")

Clicks growth: 71.43%
Impressions growth: 71.49%
CTR current/prior: 0.8929% / 0.8933%
Position improvement: 3.7
AI impressions growth: 161.47%


In [3]:
page_one_opportunities = sorted(
    [p for p in pages if p['position'] <= 12 and p['ctr'] < 0.02],
    key=lambda p: (p['position'], -p['impressions'])
)
for p in page_one_opportunities:
    print(f"{p['page']}: {p['impressions']:,} impressions, {p['ctr']:.2%} CTR, position {p['position']:.1f}")

National Museum: 69 impressions, 0.00% CTR, position 7.0
Venue finder: 290 impressions, 1.40% CTR, position 7.6
Event fabrication SG: 1,108 impressions, 0.50% CTR, position 10.7
Marina Bay Sands: 485 impressions, 0.80% CTR, position 11.0


In [4]:
top_two_ai_share = (ai_pages['Booth design/build'] + ai_pages['Event fabrication SG']) / gsc['ai_current']
print(f"Top two AI pages share: {top_two_ai_share:.2%}")

Top two AI pages share: 48.95%


## Takeaways

1. 先对齐生产分支，再升级依赖与响应头；否则优化可能建立在过期源码上。
2. National Museum、Venue Finder、Marina Bay Sands 和 Event Fabrication 是首轮 CTR 实验对象。
3. Booth 低 CTR 的主因是平均排名与页面簇意图重叠，不应仅靠改 title。
4. AI optimization 应增加可直接引用的答案块、服务边界、决策表、案例证据和复核日期；AI report 当前不能证明点击或询盘。